In [1]:
import json
import os
import sys
import time
from datetime import date, datetime

import numpy as np
import pandas as pd
import polars as pl
import pyarrow

In [2]:
data_2019 = "../../novus/matchingnemo/scratch/safegraph_data/Weekly Patterns/2019_Weekly_Patterns/"
example = os.listdir(data_2019)[2]

In [3]:
city = 'Portland'

In [4]:
cols_to_read = [
    "safegraph_place_id",
    "location_name",
    "street_address",
    "city",
    "region",
    "postal_code",
    "iso_country_code",
    "date_range_start",
    "raw_visit_counts",
    "raw_visitor_counts",
    "visits_by_day",
    "visits_by_each_hour",
    "poi_cbg",
    "visitor_home_cbgs",
    "visitor_daytime_cbgs",
    "visitor_country_of_origin",
    "distance_from_home",
    "median_dwell",
    "bucketed_dwell_times",
]
schema_overrides = {"date_range_start": pl.Datetime, "distance_from_home": pl.Int64}

In [5]:
read = (
    pl.scan_csv(os.path.join(data_2019, example), schema_overrides=schema_overrides)
    .select(cols_to_read)
    .with_columns(
        [
            pl.col("visits_by_each_hour").str.json_decode(pl.List(pl.Int64)),
            pl.col("visits_by_day").str.json_decode(pl.List(pl.Int64)),
        ]
    )
)

In [6]:
cols_to_select = [
    "safegraph_place_id",
    "city",
    "region",
    "date_range_start",
    "visits_by_each_hour",
]

In [7]:
read = (
    read.filter(
        pl.col("iso_country_code") == "US",
        pl.col("city") == city,
        pl.col("region") == "OR",
    )
    .select(cols_to_select)
    .with_columns(t=(pl.int_ranges(0, pl.col("visits_by_each_hour").list.len())))
    .explode(["t", "visits_by_each_hour"])
    .rename({"visits_by_each_hour": "visits"})
)

In [8]:
read = read.collect()

In [9]:
min(read['date_range_start'].dt.week().to_numpy())

29

In [10]:
read.head()

safegraph_place_id,city,region,date_range_start,visits,t
str,str,str,datetime[μs],i64,i64
"""sg:05b435392f714335858efe05970…","""Portland""","""OR""",2019-07-15 07:00:00,1,0
"""sg:05b435392f714335858efe05970…","""Portland""","""OR""",2019-07-15 07:00:00,0,1
"""sg:05b435392f714335858efe05970…","""Portland""","""OR""",2019-07-15 07:00:00,0,2
"""sg:05b435392f714335858efe05970…","""Portland""","""OR""",2019-07-15 07:00:00,0,3
"""sg:05b435392f714335858efe05970…","""Portland""","""OR""",2019-07-15 07:00:00,0,4


<b> Augment with FEMA data

In [11]:
# !pip install shapely geopandas

In [12]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

In [13]:
core_places_data_2019 = "../../novus/matchingnemo/scratch/ampnet_data/safegraph_POIs/2019.parquet"

In [14]:
gdf = gpd.read_file(r"../../novus/matchingnemo/scratch/safegraph_data/Weekly Patterns/Digital_Twins_Analysis/temporary_stash_very_heavy/entire_or_structures_clip.gpkg")
gdf_subset = gdf[
    [
        "BUILD_ID",
        "OCC_CLS",
        "PRIM_OCC",
        "SQMETERS",
        "SQFEET",
        "CENSUSCODE",
        "UUID",
        "geometry",
    ]
]

gdf_subset.columns = gdf_subset.columns.str.lower()
gdf_nonresidential = gdf_subset[gdf_subset["occ_cls"] != "Residential"]


places_data = pd.read_parquet(core_places_data_2019)
places_centroid = gpd.GeoDataFrame(
    places_data,
    geometry=gpd.points_from_xy(places_data.longitude, places_data.latitude),
    crs="EPSG:4326",
)

In [15]:
matches = gpd.sjoin(
    gdf_nonresidential, places_centroid, predicate="contains", how="left"
)

In [23]:
matches.groupby("safegraph_place_id")["build_id"].nunique().sort_values(ascending=False)

safegraph_place_id
nan                                    195206
sg:501dbd1baf014af4b9a0e9aa155a5378         2
sg:a629e4a6581e4dcb8f28c0c679400ea1         1
sg:a62067cc9050447a88441fddc2adb417         1
sg:a623f95cbd284ba8bc4acd3f42913e73         1
                                        ...  
sg:5141bc8aa8f7409ea7bb5be4cf12dcb3         1
sg:5143194b3fbc45259556eb6b28e68b1b         1
sg:51447fc3dcc648858193c315362645d9         1
sg:5149b075aafa4bf0880c06bbc795d927         1
sg:ffff0398653347b29cc883d10d592262         1
Name: build_id, Length: 57516, dtype: int64

In [16]:
read_pd = read.to_pandas()
matches["safegraph_place_id"] = matches["safegraph_place_id"].astype(str)
read_pd["safegraph_place_id"] = read_pd["safegraph_place_id"].astype(str)
outfile = matches.merge(read_pd, on="safegraph_place_id", how="left")

In [28]:
cols_in_csv = [
    "build_id",
    "occ_cls",
    "prim_occ",
    "sqmeters",
    "sqfeet",
    "censuscode",
    "uuid",
    "safegraph_place_id",
    "date_range_start",
    "t",
    "visits"
]

agg_dict = {item: 'first' for item in cols_in_csv}
agg_dict['visits'] = 'sum'
del(agg_dict['t'])
del(agg_dict['build_id'])
outfile = outfile.loc[:,cols_in_csv]

outfile2 = outfile.groupby(['build_id', 't']).agg(agg_dict).reset_index()


In [19]:
# outfile.write_csv("../../novus/matchingnemo/scratch/ampnet_data/test.csv")